 # Sentiment Analysis

##### Preprocessed Data

In [1]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data()
x_train[0][:10]

2023-04-08 09:14:14.467970: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]

In [2]:
word_index = tf.keras.datasets.imdb.get_word_index()
list(word_index.items())[:10]

[('fawn', 34701),
 ('tsukino', 52006),
 ('nunnery', 52007),
 ('sonja', 16816),
 ('vani', 63951),
 ('woods', 1408),
 ('spiders', 16115),
 ('hanging', 2345),
 ('woody', 2289),
 ('trawling', 52008)]

In [3]:
id_to_word = {id_ + 3: word for word, id_ in word_index.items()}
for id_, token in enumerate(("<pad>", "<sos>", "<unk>")):
    id_to_word[id_] = token
list(id_to_word.items())[:10]

[(34704, 'fawn'),
 (52009, 'tsukino'),
 (52010, 'nunnery'),
 (16819, 'sonja'),
 (63954, 'vani'),
 (1411, 'woods'),
 (16118, 'spiders'),
 (2348, 'hanging'),
 (2292, 'woody'),
 (52011, 'trawling')]

In [4]:
" ".join([id_to_word[id_] for id_ in x_train[0]])

"<sos> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert redford's is an amazing actor and now the same being director norman's father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for retail and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also congratulations to the two little boy's that played the part's of norman and paul they were just brilliant children are often left out of the praising list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and shou

##### Restart from the Raw Data

In [5]:
import tensorflow_datasets as tfds

datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)
info

2023-04-08 09:14:20.000496: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-08 09:14:20.104614: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-08 09:14:20.104842: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-08 09:14:20.106443: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_path='/home/yuncong/tensorflow_datasets/imdb_reviews/plain_text/1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    splits={
        'test': <SplitInfo num_examples=25000, num_shards

In [6]:
info.splits["train"].num_examples

25000

In [7]:
datasets

{'train': <PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 'test': <PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 'unsupervised': <PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>}

In [8]:
for instance in datasets["train"].take(5):
    print(instance)

(<tf.Tensor: shape=(), dtype=string, numpy=b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.">, <tf.Tensor: shape=(), dtype=int64, numpy=0>)
(<tf.Tensor: shape=(), dtype=string, numpy=b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on

2023-04-08 09:14:21.108759: W tensorflow/core/kernels/data/cache_dataset_ops.cc:856] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [9]:
# Test tf.strings.regex_replace()
tf.strings.regex_replace([b"Well, I can't<br />, this is a test"], b"<br\s*/?>", b" ")

<tf.Tensor: shape=(1,), dtype=string, numpy=array([b"Well, I can't , this is a test"], dtype=object)>

In [10]:
tf.strings.split([b"This is a test"])

<tf.RaggedTensor [[b'This', b'is', b'a', b'test']]>

In [11]:
from typing import Tuple


def preprocess(
    x_batch: tf.data.Dataset, y_batch: tf.data.Dataset
) -> Tuple[tf.data.Dataset, tf.data.Dataset]:
    x_batch = tf.strings.substr(x_batch, 0, 300)
    x_batch = tf.strings.regex_replace(x_batch, b"<br\\s*/?>", b" ")
    x_batch = tf.strings.regex_replace(x_batch, b"[^a-zA-Z]", b" ")
    x_batch = tf.strings.split(x_batch)
    return x_batch.to_tensor(default_value=b"<pad>"), y_batch

In [12]:
from collections import Counter

vocab = Counter()
for x_batch, y_batch in datasets["train"].batch(32).map(preprocess):
    for review in x_batch:
        vocab.update(list(review.numpy()))

vocab

Counter({b'This': 6677,
         b'was': 14954,
         b'an': 5185,
         b'absolutely': 460,
         b'terrible': 519,
         b'movie': 15035,
         b'Don': 314,
         b't': 8145,
         b'be': 5890,
         b'lured': 6,
         b'in': 18973,
         b'by': 4717,
         b'Christopher': 99,
         b'Walken': 32,
         b'or': 3305,
         b'Michael': 333,
         b'Ironside': 6,
         b'Both': 66,
         b'are': 5666,
         b'great': 2408,
         b'actors': 1102,
         b'but': 7724,
         b'this': 18492,
         b'must': 800,
         b'simply': 394,
         b'their': 1818,
         b'worst': 1103,
         b'role': 544,
         b'history': 343,
         b'Even': 348,
         b'acting': 1905,
         b'could': 1680,
         b'not': 6326,
         b'redeem': 13,
         b's': 13884,
         b'ridiculous': 231,
         b'storyline': 248,
         b'is': 25719,
         b'early': 443,
         b'nineties': 18,
         b'US': 131,
     

In [13]:
vocab.most_common(3)

[(b'<pad>', 224494), (b'the', 61156), (b'a', 38569)]

In [14]:
len(vocab)

49739

In [15]:
size_vocab = 10_000
vocab_trunc = [word for word, count in vocab.most_common()[:size_vocab]]
vocab_trunc[:10]

[b'<pad>', b'the', b'a', b'of', b'and', b'I', b'to', b'is', b'it', b'in']

In [16]:
words_tensor = tf.constant(vocab_trunc)
words_tensor

<tf.Tensor: shape=(10000,), dtype=string, numpy=
array([b'<pad>', b'the', b'a', ..., b'blessed', b'PEOPLE', b'elevator'],
      dtype=object)>

In [17]:
ids_words = tf.range(len(vocab_trunc), dtype=tf.int64)
ids_words

<tf.Tensor: shape=(10000,), dtype=int64, numpy=array([   0,    1,    2, ..., 9997, 9998, 9999])>

In [18]:
vocab_init = tf.lookup.KeyValueTensorInitializer(words_tensor, ids_words)
vocab_init

In [19]:
num_oov_buckets = 1_000
table = tf.lookup.StaticVocabularyTable(vocab_init, num_oov_buckets)
table

In [20]:
table.lookup(tf.constant([b"This move was faaaaaantastic".split()]))

<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[   24,   943,    13, 10053]])>

In [21]:
def encode_words(
    x_batch: tf.data.Dataset, y_batch: tf.data.Dataset
) -> Tuple[tf.data.Dataset, tf.data.Dataset]:
    return table.lookup(x_batch), y_batch


train_set = datasets["train"].batch(32).map(preprocess).map(encode_words).prefetch(1)

In [22]:
size_embed = 128
model = tf.keras.Sequential(
    [
        tf.keras.layers.Embedding(
            size_vocab + num_oov_buckets, size_embed, input_shape=[None]
        ),
        tf.keras.layers.GRU(128, return_sequences=True),
        tf.keras.layers.GRU(128),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 128)         1408000   
                                                                 
 gru (GRU)                   (None, None, 128)         99072     
                                                                 
 gru_1 (GRU)                 (None, 128)               99072     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1,606,273
Trainable params: 1,606,273
Non-trainable params: 0
_________________________________________________________________


In [23]:
import time
from pathlib import Path

root_logdir = Path().absolute() / "logs"

%load_ext tensorboard
%tensorboard --logdir=./logs --port=6006

Launching TensorBoard...

In [24]:
log_dir = root_logdir / time.strftime("run_%Y_%m_%d-%H_%M_%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir, histogram_freq=1)

In [25]:
history = model.fit(train_set, epochs=5, callbacks=[tensorboard_callback])

Epoch 1/5


2023-04-08 09:14:33.434822: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8401


782/782 [==============================] - 12s 12ms/step - loss: 0.6792 - accuracy: 0.5560
Epoch 2/5
782/782 [==============================] - 9s 12ms/step - loss: 0.4684 - accuracy: 0.7760
Epoch 3/5
782/782 [==============================] - 9s 12ms/step - loss: 0.3170 - accuracy: 0.8662
Epoch 4/5
782/782 [==============================] - 9s 12ms/step - loss: 0.2210 - accuracy: 0.9166
Epoch 5/5
782/782 [==============================] - 9s 12ms/step - loss: 0.1699 - accuracy: 0.9369


In [26]:
test_set = datasets["test"].batch(32).map(preprocess).map(encode_words).prefetch(1)
model.evaluate(test_set)

782/782 [==============================] - 5s 5ms/step - loss: 0.7799 - accuracy: 0.7411


[0.7799167037010193, 0.7410799860954285]